# 线性回归在设备维护时间预测中的实训案例



## 1. 项目背景


在工业生产领域，设备的稳定运行是保障生产效率的核心环节。

随着智能制造的发展，预测性维护已成为降低设备故障率、减少停机时间的关键技术。

通过分析设备的运行数据（如使用年限、运行时长、温度波动等），建立数学模型预测设备所需的维护时间，

能够帮助企业合理规划维护资源、降低运营成本。

本实训以工业设备维护数据为研究对象，旨在通过线性回归模型探索设备特征与维护时间的关联关系，

掌握从数据加载、分析到模型构建、评估的完整流程，为实际生产中的维护决策提供数据支持。

完成本课程后，你将能够：

*   **理解** 线性回归模型在机器维护时间预测中的应用场景和基本原理。
*   **掌握** 使用 Pandas 进行基本数据加载、探索与预处理的方法。
*   **学会** 使用 Scikit-learn 库构建、训练和评估一个线性回归模型。
*   **运用** 训练好的模型对新的机器数据进行维护时间预测。
*   **评估** 模型性能，并解读模型结果。



## 2. 数据集说明



### 2.1 数据集来源



Industrial_Equipment_Maintenance

包含 1000 条设备记录

涵盖多种工业设备类型（如离心泵、涡轮空压机、CNC 主轴驱动系统等）


### 2.2 核心字段



| 字段名称                        | 说明                             | 数据类型  |
|-----------------------------|--------------------------------|-------|
| Machine_ID                  | 设备唯一标识                        | 字符串 |
| Machine_Name                | 设备类型（如"Turbine Air Compressor"） | 字符串 |
| Model                       | 设备型号(如"TAC-10X")              | 字符串 |
| Age_Years                   | 设备使用年限 (年)                   | 数值型 |
| Operating_Hours_Last_Month  | 上月运行小时数                       | 数值型 |
| Maintenance_History_Count   | 历史维护次数                        | 数值型 |
| Temperature_Fluctuation     | 温度波动值 (℃)                     | 数值型 |
| Predicted_Maintenance_Time_Hours | 目标变量：预测维护时间(小时)        | 数值型 |


## 3. 数据分析

### 3.1 数据加载与初步探索



通过 Python 的pandas库加载数据集，查看数据结构、缺失值及基本统计特征，理解数据分布规律。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 设置图表样式，确保中文显示正常
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei']  # 显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 正常显示负号

# 从指定路径加载数据集
df = pd.read_csv('/home/jovyan/work/datasets/688b0e4e4e03dbf50518a121-momodel/industrial_equipment_maintenance.csv')

# 查看数据集基本信息
print("数据集前5行：")
print(df.head())
print("\n数据集基本信息：")
df.info()

### 3.2 相关性分析

通过皮尔逊相关系数计算数值型字段间的关联强度，并用热力图可视化，识别对目标变量（维护时间）影响较大的特征

In [ ]:
# ---------------------- 1. 相关性热力图 ----------------------
# 计算所有数值列之间的皮尔逊相关系数

numeric_df = df.select_dtypes(include=['number'])  # 筛选出所有数值型列
correlation_matrix = numeric_df.corr()  # 基于数值列计算相关性

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix, 
    annot=True, 
    cmap='coolwarm', 
    fmt=".2f", 
    linewidths=.5
)
plt.title('图2.1：特征与目标变量相关性热力图', fontsize=16, pad=20)
plt.xlabel('变量', fontsize=12)
plt.ylabel('变量', fontsize=12)
plt.xticks(rotation=45, ha='right')  # 旋转x轴标签
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()





分析结论：

- 若Operating_Hours_Last_Month（上月运行小时数）与Predicted_Maintenance_Time_Hours（维护时间）的相关系数接近 0.5，表明两者存在较强正相关；

- Temperature_Fluctuation（温度波动）也可能与维护时间呈正相关（设备温度波动越大，维护需求越高）。

## 4. 模型训练

### 4.1 数据准备

特征变量（X）：选择对维护时间可能有影响的字段，如Age_Years、Operating_Hours_Last_Month、Maintenance_History_Count、Temperature_Fluctuation；

目标变量（y）：Predicted_Maintenance_Time_Hours（需预测的维护时间）；

数据集划分：按 8:2 比例划分为训练集（用于模型学习）和测试集（用于模型评估）。

In [ ]:
# 提取所需字段（特征和目标变量）
features = ['Age_Years', 'Operating_Hours_Last_Month', 'Maintenance_History_Count', 'Temperature_Fluctuation']
X = df[features]
y = df['Predicted_Maintenance_Time_Hours']

print("\n特征 (X) 的前5行：")
print(X.head())
print("\n目标变量 (y) 的前5行：")
print(y.head())

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n训练集特征形状：{X_train.shape}")
print(f"测试集特征形状：{X_test.shape}")
print(f"训练集目标形状：{y_train.shape}")
print(f"测试集目标形状：{y_test.shape}")



### 4.2 线性回归模型构建

#### 4.2.1 多特征线性回归

使用sklearn的LinearRegression构建模型，学习特征与目标变量的线性关系：

In [ ]:
# 训练多特征线性回归模型
model = LinearRegression()
model.fit(X_train, y_train)

print("\n模型训练完成！")
print("\n模型系数 (Weights):")
for feature, coef in zip(features, model.coef_):
    print(f"{feature}: {coef:.4f}")
print(f"\n模型截距 (Intercept): {model.intercept_:.4f}")

# 生成测试集预测值（用于后续可视化）
y_pred = model.predict(X_test)

模型解读：

系数为正表明特征与维护时间呈正相关

如Operating_Hours_Last_Month系数为 0.03，意味着上月运行小时数每增加 100 小时，维护时间平均增加 3 小时；

截距表示所有特征为 0 时的基准维护时间。

#### 4.2.2 单特征线性回归（以运行小时数为例）

为直观展示线性关系，单独使用Operating_Hours_Last_Month训练模型，并绘制回归线：

In [ ]:
# ---------------------- 2. 单特征线性回归可视化 ----------------------
X_single = df[['Operating_Hours_Last_Month']]
y_single = df['Predicted_Maintenance_Time_Hours']

# 划分单特征的训练集和测试集
X_train_single, X_test_single, y_train_single, y_test_single = train_test_split(
    X_single, y_single, test_size=0.2, random_state=42
)

# 训练单特征模型
model_single = LinearRegression()
model_single.fit(X_train_single, y_train_single)

# 绘制散点图和回归线
plt.figure(figsize=(12, 7))
sns.scatterplot(
    x=X_train_single['Operating_Hours_Last_Month'], 
    y=y_train_single,
    label='实际训练数据点', 
    alpha=0.7, 
    color='steelblue', 
    s=80
)

# 生成回归线数据（转换为DataFrame避免特征名警告）
x_range = pd.DataFrame({
    'Operating_Hours_Last_Month': np.linspace(
        X_train_single['Operating_Hours_Last_Month'].min() * 0.95,
        X_train_single['Operating_Hours_Last_Month'].max() * 1.05, 
        100
    )
})
y_pred_line = model_single.predict(x_range)

# 绘制回归线
plt.plot(
    x_range, 
    y_pred_line, 
    color='red', 
    linestyle='-', 
    linewidth=2.5,
    label=f'线性回归线: Y = {model_single.coef_[0]:.2f}X + {model_single.intercept_:.2f}'
)

# 图表标签与注释
plt.title('图3.1：单特征线性回归示例 — 揭示“最佳拟合线”', fontsize=16, pad=20)
plt.xlabel('月运行小时数 (Operating_Hours_Last_Month)', fontsize=12)
plt.ylabel('预测维护时间 (小时)', fontsize=12)
plt.legend(fontsize=10, loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.annotate(
    '这条红线是模型学习到的“最佳拟合线”，\n它代表了月运行小时数与维护时间之间的线性关系。',
    xy=(x_range.mean().values[0], y_pred_line.mean()),
    xytext=(0.6, 0.2),
    textcoords='axes fraction',
    arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8),
    fontsize=11, 
    color='darkgreen',
    bbox=dict(boxstyle="round,pad=0.3", fc="yellow", ec="darkgreen", lw=1, alpha=0.8)
)
plt.show()


## 5.模型评估与可视化

### 5.1 预测结果对比

通过测试集预测值与实际值的对比，评估模型准确性：

In [ ]:
# ---------------------- 3. 预测值与实际值对比图 ----------------------
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=y_test, 
    y=y_pred, 
    alpha=0.7, 
    color='darkorange', 
    s=80, 
    label='预测 vs. 实际数据点'
)

# 添加 y=x 参考线（理想预测线）
min_val = min(y_test.min(), y_pred.min()) * 0.9
max_val = max(y_test.max(), y_pred.max()) * 1.1
plt.plot(
    [min_val, max_val], 
    [min_val, max_val], 
    color='red', 
    linestyle='--', 
    linewidth=2, 
    label='理想预测线 (y=x)'
)

plt.title('图4.1：模型预测值与实际值对比图', fontsize=16, pad=20)
plt.xlabel('实际维护时间 (小时) - True Values', fontsize=12)
plt.ylabel('模型预测维护时间 (小时) - Predicted Values', fontsize=12)
plt.legend(fontsize=10, loc='upper left')
plt.grid(True, linestyle=':', alpha=0.5)
plt.axis('equal')  # 确保x和y轴比例一致
plt.tight_layout()
plt.show()

解读：数据点越接近红色虚线（y=x），表明预测越准确。

### 5.2 残差分析

残差（实际值 - 预测值）用于检验模型是否捕捉了所有线性关系：

- 残差散点图：若残差随机分布在 0 轴附近，无明显模式，表明模型假设合理；
- 残差直方图：若残差近似正态分布，表明模型误差符合线性回归假设。

In [ ]:
# ---------------------- 4. 残差散点图 ----------------------
# 计算残差（真实值 - 预测值）
residuals = y_test - y_pred

plt.figure(figsize=(12, 7))
sns.scatterplot(
    x=y_pred, 
    y=residuals, 
    alpha=0.7, 
    color='purple', 
    s=80
)
plt.axhline(
    y=0, 
    color='red', 
    linestyle='--', 
    linewidth=2, 
    label='零残差线'
)

plt.title('图4.2：残差图 — 预测值与残差的关系', fontsize=16, pad=20)
plt.xlabel('模型预测值 (Predicted Values)', fontsize=12)
plt.ylabel('残差 (真实值 - 预测值)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle=':', alpha=0.5)

plt.annotate(
    '理想情况下，残差应随机分布在零轴上下，\n没有明显的模式（如U型、扇形）。\n这表明模型很好地捕捉了线性关系。',
    xy=(y_pred.mean(), residuals.min() * 0.8),
    xytext=(0.05, 0.85),
    textcoords='axes fraction',
    arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8),
    fontsize=11, 
    color='darkblue',
    bbox=dict(boxstyle="round,pad=0.3", fc="lightblue", ec="darkblue", lw=1, alpha=0.8)
)
plt.show()

In [ ]:
# ---------------------- 5. 残差分布直方图 ----------------------
plt.figure(figsize=(12, 7))
sns.histplot(
    residuals, 
    kde=True, 
    color='teal', 
    bins=20, 
    edgecolor='black', 
    alpha=0.7
)
plt.axvline(
    x=0, 
    color='red', 
    linestyle='--', 
    linewidth=2, 
    label='残差均值应为零'
)

plt.title('图4.3：残差分布直方图 — 检查正态性与均值', fontsize=16, pad=20)
plt.xlabel('残差 (Residuals)', fontsize=12)
plt.ylabel('频率 / 密度', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle=':', alpha=0.5)

plt.annotate(
    '理想的残差分布应近似呈钟形曲线（正态分布），\n且其中心（均值）应接近于零。',
    xy=(residuals.mean(), plt.gca().get_ylim()[1] * 0.8),
    xytext=(0.05, 0.85),
    textcoords='axes fraction',
    arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8),
    fontsize=11, 
    color='darkred',
    bbox=dict(boxstyle="round,pad=0.3", fc="mistyrose", ec="darkred", lw=1, alpha=0.8)
)
plt.show()

## 6.总结

本实训构建的模型可用于：

基于设备运行数据提前预测维护时间，辅助企业制定维护计划；
识别对维护时间影响最大的因素（如运行小时数、温度波动），针对性优化设备运行条件。

思考题：

Q.1 皮尔逊相关系数的取值范围是什么？如何通过热力图判断特征与目标变量的相关性强弱?  
A.1 取值范围是[-1，1]。热力图中颜色越接近红／蓝（正值／负值）、数值越接近 ±1，相关性越强。

Q.2 线性回归模型的系数（coef）表示什么含义？若某特征系数为0.05，说明什么？  
A.2 表示特征每增加 1单位，目标变量的平均变化量。系数 0.05说明该特征每增加1单位，维护时间平均增加 0.05小时。

Q.3 为什么要将数据集划分为训练集和测试集？测试集的作用是什么？  
A.3 避免模型过拟合训练数据。测试集用于评估模型在新数据上的泛化能力。

Q.4 残差图中，若残差呈现明显的U型分布，说明模型存在什么问题?  
A.4 说明模型未捕捉到数据中的非线性关系，假设的线性关系不成立。

Q.5 单特征回归与多特征回归的区别是什么？哪种更可能有更好的预测效果?  
A.5 单特征仅用一个变量建模，多特征综合多个变量。通常多特征回归更优（若特征选择合理）。